# 03 — Pipeline Spark Streaming + Kafka

Este notebook documenta e implementa el flujo en vivo del proyecto:

`datos historicos NYC Taxi -> Kafka -> Spark Streaming -> agregacion por zona/30 min -> modelo GBT -> predicciones`

La idea es simular tiempo real usando datos historicos. Si se emiten los mismos viajes historicos con los mismos timestamps, las features temporales y espaciales son las mismas, asi que las predicciones deben ser reproducibles independientemente de si hoy es lunes o jueves. El dia actual de ejecucion no entra en el modelo; entra el `pickup_datetime` del mensaje historico.

## Rutas usadas

- Kafka local: `kafka_2.12-3.7.0/`
- Topic: `taxi-trips`
- Productor: `producer/taxi_producer.py`
- Consumidor streaming: `streaming/taxi_consumer.py`
- Modelo usado por la guia: `models/gbt_taxi`
- Salida streaming: `streaming_output/`
- Checkpoint Spark: `streaming_checkpoint/`

El modelo `models/gbt_taxi` espera estas features:

```text
zone_lon, zone_lat, hour, dayofweek,
is_weekend, is_rush_hour, is_late_night,
lag_1, lag_2, lag_48
```

Por eso el consumidor streaming genera esas columnas antes de llamar a `modelo.transform(...)`.

## Terminales que hay que abrir

Ejecuta estos pasos en terminales separadas y en este orden. Esta es la logica definitiva del proyecto: primero dejamos Spark Streaming escuchando y despues arrancamos el productor para simular llegadas en vivo.

```text
T1: Zookeeper
T2: Kafka broker
T3: Consumidor Spark Streaming
T4: Productor Kafka
```

Con este orden, `startingOffsets="latest"` es lo adecuado: el consumidor lee los viajes nuevos a medida que el productor los publica.

### Terminal 1 — Zookeeper

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/zookeeper-server-start.sh config/zookeeper.properties
```

Espera a ver que Zookeeper queda escuchando en el puerto `2181`.

### Terminal 2 — Kafka broker

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/kafka-server-start.sh config/server.properties
```

Espera a ver un mensaje parecido a `[KafkaServer id=0] started`.

### Terminal 2 o una terminal auxiliar — crear el topic

Solo hace falta crearlo la primera vez. Si ya existe, Kafka avisara y no pasa nada.

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/kafka-topics.sh   --create   --topic taxi-trips   --bootstrap-server localhost:9092   --partitions 1   --replication-factor 1
```

Comprobar topics:

```bash
bin/kafka-topics.sh --list --bootstrap-server localhost:9092
```

### Terminal 3 — Consumidor Spark Streaming

Antes de lanzar el productor, desde la raiz del repo:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi
python streaming/taxi_consumer.py
```

Por defecto el consumidor usa `latest`, porque en la demo oficial debe arrancar antes que el productor y leer solo mensajes nuevos. Si quieres reiniciar una demo limpia, borra antes el checkpoint y la salida:

```bash
rm -rf streaming_checkpoint streaming_output
python streaming/taxi_consumer.py
```

La primera vez Spark puede descargar el conector `spark-sql-kafka-0-10_2.12:3.5.4`. Cuando arranque, cada batch mostrara las zonas con mayor prediccion.

### Terminal 4 — Productor Kafka

Cuando el consumidor ya este arrancado, desde la raiz del repo:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi
python producer/taxi_producer.py --limit 50000 --sleep 0.001 --filter-nyc
```

Para demo, `--limit 50000` evita enviar los 14 millones de viajes. Para enviar todo:

```bash

python producer/taxi_producer.py --limit 0 --sleep 0 --filter-nyc
```

El productor publica mensajes JSON como:

```json
{"pickup_datetime": "2009-01-04 18:23:00", "start_lon": -73.991, "start_lat": 40.748}
```

## Verificacion opcional de Kafka

Para ver tres mensajes publicados en el topic:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/kafka-console-consumer.sh   --topic taxi-trips   --bootstrap-server localhost:9092   --from-beginning   --max-messages 3
```

## Leer resultados guardados

Cuando el consumidor haya escrito datos en `streaming_output/`, puedes leerlos desde Spark:

In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("nyc-taxi-streaming-results")
    .master("local[*]")
    .getOrCreate()
)

df_resultados = spark.read.parquet("../streaming_output")
df_resultados.printSchema()
df_resultados.orderBy("window_start", "prediction", ascending=[True, False]).show(20, truncate=False)

root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- zone_lon: double (nullable = true)
 |-- zone_lat: double (nullable = true)
 |-- trip_count: double (nullable = true)
 |-- prediction: double (nullable = true)
 |-- error: double (nullable = true)
 |-- abs_error: double (nullable = true)

+-------------------+-------------------+--------+--------+----------+------------------+-------------------+------------------+
|window_start       |window_end         |zone_lon|zone_lat|trip_count|prediction        |error              |abs_error         |
+-------------------+-------------------+--------+--------+----------+------------------+-------------------+------------------+
|2009-01-26 23:00:00|2009-01-26 23:30:00|-73.98  |40.75   |365.0     |490.87359744151274|125.87359744151274 |125.87359744151274|
|2009-01-26 23:00:00|2009-01-26 23:30:00|-73.99  |40.76   |297.0     |489.78017045521756|192.78017045521756 |192.78017045521756|
|2009-01-26 23:

In [4]:
from pyspark.ml.evaluation import RegressionEvaluator

# R2
evaluator_r2 = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="r2"
)

r2 = evaluator_r2.evaluate(df_resultados)

# RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="rmse"
)

rmse = evaluator_rmse.evaluate(df_resultados)

# MAE
evaluator_mae = RegressionEvaluator(
    labelCol="trip_count",
    predictionCol="prediction",
    metricName="mae"
)

mae = evaluator_mae.evaluate(df_resultados)

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")

R²: 0.9004
RMSE: 46.3611
MAE: 13.5129


## Version resumida del consumidor

El script real esta en `streaming/taxi_consumer.py`. Esta celda deja visible la logica principal dentro del notebook para que el flujo quede documentado.

In [ ]:
# Resumen conceptual del consumidor:
# 1. Leer mensajes JSON desde Kafka.
# 2. Convertir pickup_datetime a timestamp.
# 3. Filtrar coordenadas validas de NYC.
# 4. Crear zone_lon y zone_lat con grid de 0.01 grados.
# 5. Agregar en streaming por window(pickup_datetime, '30 minutes') + zona.
# 6. Usar outputMode('update') para ver ventanas que se van actualizando.
# 7. Crear hour, dayofweek, is_weekend, is_rush_hour, is_late_night.
# 8. Crear lag_1, lag_2 y lag_48 usando historial por zona.
# 9. Cargar models/gbt_taxi con PipelineModel.load(...).
# 10. Aplicar modelo.transform(features_df).
# 11. Mostrar top zonas y guardar en streaming_output/.

## Si algo falla

- Si aparece `Could not find class org.apache.spark.sql.kafka...`, ejecuta otra vez el consumidor con internet; Spark necesita descargar el paquete Kafka.
- Si cambias el codigo del consumidor o quieres empezar una demo limpia:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi
rm -rf streaming_checkpoint streaming_output
```

- Si el topic tiene mensajes antiguos y quieres reiniciar la demo limpia:

```bash
cd /home/alumno/Desktop/bigdata-nyc-taxi/kafka_2.12-3.7.0
bin/kafka-topics.sh --delete --topic taxi-trips --bootstrap-server localhost:9092
bin/kafka-topics.sh --create --topic taxi-trips --bootstrap-server localhost:9092 --partitions 1 --replication-factor 1
```

- Si por accidente lanzaste primero el productor y despues el consumidor, puedes repetir la demo limpia borrando el topic o relanzando el productor con el consumidor ya abierto. No es el flujo recomendado para la presentacion.
